# Student Attendance Prediction System

Predicts a student's likely **future attendance percentage** and classifies
their **risk** of falling below the minimum required attendance, from four
lecture counts: lectures delivered, lectures attended, medical leave and
duty leave. (Internal marks and other academic scores are deliberately not
used - they don't affect attendance.)

**Pipeline**

```
STUDENT DATA
     |
Data Preprocessing
     |
Feature Selection
     |
Train/Test Split
     |
Machine Learning Model   (Logistic Regression + Linear Regression)
     |
Model Evaluation
     |
Trained ML Model
     |
Prediction System
     |
Student Input -> Prediction -> Attendance Status / Expected % / Risk Level
```

No real student records were available, so this notebook builds and
validates the pipeline on a **synthetic but realistically-correlated**
dataset (Section 1). To use real records, see the notes at the end.

> Run this notebook from the project folder -- it imports
> `attendance_predictor.py`, which lives next to it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from attendance_predictor import (
    INPUT_FIELDS, FEATURES, REQUIRED_ATTENDANCE, attendance_stats,
    generate_synthetic_dataset, preprocess, feature_selection_report,
    split_and_scale, train_models, evaluate_classifier, evaluate_regressor,
    save_trained_system, load_trained_system,
    predict_attendance, display_prediction,
)

print("Required attendance threshold:", REQUIRED_ATTENDANCE, "%")
print("Inputs a student enters:", INPUT_FIELDS)
print("Features the models learn from:", FEATURES)

## 1. Student Data

A student records four whole numbers of lectures:

| Input | Meaning |
|---|---|
| `Lectures_Delivered` | Total lectures held so far |
| `Lectures_Attended` | Lectures the student was present for |
| `Medical_Leave` | Lectures missed on approved medical leave |
| `Duty_Leave` | Lectures missed on approved duty leave |

The attendance percentage is then exact arithmetic:
`(attended + medical + duty) / delivered * 100` (set `COUNT_MEDICAL_LEAVE` /
`COUNT_DUTY_LEAVE` in `attendance_predictor.py` if your institution does not
count a leave type). The models learn from four derived features:
`Attendance_Pct`, `Medical_Pct`, `Duty_Pct` and `Lectures_Delivered`.

Each synthetic student is generated from a shared "engagement" trait that
moves how often they attend. Medical and duty leave are independent of it
(illness and college events happen to everyone). Noise keeps the target from
being perfectly predictable - real students can't be predicted perfectly
either.

Two targets are produced for each student: `Future_Attendance` (the next
period's attendance %, used by the regression) and `Sufficient_Attendance`
(1 if that reaches the required %, used by the classifier).

In [ ]:
df = generate_synthetic_dataset()
print("Shape:", df.shape)
print(df.head(8).to_string(index=False))

## 2. Data Preprocessing

Drops rows with no target, fills any missing inputs with the column median,
and clips every input to a valid range (percentages to 0-100, and so on).
The synthetic data has none of these problems, but real attendance
registers usually do (blank cells, stray values like 105%), so the step
stays in.

In [ ]:
df = preprocess(df)
print("Missing values remaining:", int(df[FEATURES].isnull().sum().sum()))
print(df[FEATURES + ["Future_Attendance"]].describe().round(2).to_string())

## 3. Feature Selection

Rather than dropping inputs, this checks each feature with an F-test against both
targets. `Attendance_Pct` dominates, and `Medical_Pct` and `Duty_Pct` are
significant (p < 0.001). `Lectures_Delivered` is not significant on its own -
it is kept as context (a percentage over 20 lectures is less reliable than
over 120), but expect it to matter little.

In [ ]:
plt.figure(figsize=(6, 4.5))
corr = df[FEATURES + ["Future_Attendance"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Feature Correlation")
plt.tight_layout()
plt.show()

print()
print(feature_selection_report(df).to_string(index=False))

## 4. Train/Test Split

An 80/20 split, stratified on `Sufficient_Attendance` so both sets keep the
same class balance, then the features are standardized (mean 0, unit
variance) -- this matters for logistic regression's regularization and
keeps both models' coefficients on a comparable scale. The scaler is fit on
the training rows only, so nothing from the test set leaks into training.

In [ ]:
X_train, X_test, yc_train, yc_test, yr_train, yr_test, scaler = split_and_scale(df)

print(f"Train rows: {X_train.shape[0]}  |  Test rows: {X_test.shape[0]}")
print("Share 'sufficient' (train):", round(np.bincount(yc_train)[1] / len(yc_train), 3))
print("Share 'sufficient' (test): ", round(np.bincount(yc_test)[1] / len(yc_test), 3))

## 5. Machine Learning Model

Two models share the same four (scaled) features:

- **Logistic Regression** -- classifies whether future attendance will be
  *sufficient*, and its predicted probability is what later drives the
  LOW / MEDIUM / HIGH risk level. This is the model the brief asks for: a
  simple, well-understood baseline that's easy to explain to a
  non-technical reader (see Section 7).
- **Linear Regression** -- predicts the actual future attendance
  percentage, since "Predicted Attendance: 71%" needs a number, not just a
  class.

In [ ]:
clf, reg = train_models(X_train, yc_train, yr_train)
print("Logistic Regression and Linear Regression trained on", X_train.shape[0], "students.")

## 6. Model Evaluation

Metrics for the Logistic Regression classifier -- accuracy, precision,
recall, F1 and the confusion matrix -- on the held-out test set, followed
by the Linear Regression's error on the continuous attendance prediction.

In [ ]:
metrics, cm, report_text, fig = evaluate_classifier(clf, X_test, yc_test)

for k, v in metrics.items():
    print(f"{k.capitalize():10s}: {v:.3f}")
print()
print(report_text)
plt.show()

In [ ]:
reg_metrics = evaluate_regressor(reg, X_test, yr_test)
for k, v in reg_metrics.items():
    print(f"{k.upper():6s}: {v:.3f}")

y_pred = reg.predict(X_test)
plt.figure(figsize=(5, 5))
plt.scatter(yr_test, y_pred, alpha=0.4, color="#2B4570")
lims = [0, 100]
plt.plot(lims, lims, "--", color="#AE3B2E", linewidth=1)
plt.xlabel("Actual Future Attendance (%)")
plt.ylabel("Predicted Future Attendance (%)")
plt.title("Linear Regression: Predicted vs Actual")
plt.tight_layout()
plt.show()

## 7. Interpreting the Model

One reason to start with logistic regression: every coefficient has a
direct, statable meaning. A positive coefficient means a higher value of
that feature pushes the *sufficient attendance* probability up (lower
risk); a negative one pushes it down (higher risk). The features were
standardized, so the sizes are directly comparable.

In [ ]:
coef_table = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": clf.coef_[0].round(3),
}).sort_values("Coefficient", key=abs, ascending=False)
coef_table["Effect"] = coef_table["Coefficient"].apply(
    lambda c: "higher value -> lower risk" if c > 0 else "higher value -> higher risk"
)
print(coef_table.to_string(index=False))

## 8. Trained ML Model

The fitted classifier, regressor and scaler are saved with `joblib` into the
`models/` folder, then loaded straight back -- exactly what a separate
application (or `predict.py`) does, so predictions never need retraining.

In [ ]:
for path in save_trained_system(clf, reg, scaler):
    print("Saved", path.name)

clf, reg, scaler = load_trained_system()
print("Loaded the saved model back from disk.")

## 9. Prediction System

`predict_attendance()` takes one student's four lecture counts, scales them with the
same fitted scaler, and returns the predicted attendance percentage, risk
level, status and a short recommendation. The same function powers the
command-line tool (`predict.py`); the HTML demo runs the same maths in the
browser.

In [ ]:
print(predict_attendance.__doc__)

## 10. Demo: Student Input -> Prediction

First one example, then three students side by side to show
all three risk tiers.

In [ ]:
example_student = {
    "Lectures_Delivered": 100,
    "Lectures_Attended": 62,
    "Medical_Leave": 4,
    "Duty_Leave": 2,
}

print(attendance_stats(100, 62, 4, 2))   # the exact arithmetic: percentage, can_skip, must_attend...
result = predict_attendance(example_student, clf, reg, scaler)
display_prediction(result)

In [ ]:
profiles = {
    "Below required": {"Lectures_Delivered": 100, "Lectures_Attended": 62, "Medical_Leave": 4, "Duty_Leave": 2},
    "Borderline":     {"Lectures_Delivered": 100, "Lectures_Attended": 69, "Medical_Leave": 3, "Duty_Leave": 3},
    "Strong":         {"Lectures_Delivered": 100, "Lectures_Attended": 90, "Medical_Leave": 2, "Duty_Leave": 3},
}

rows = []
for name, student in profiles.items():
    r = predict_attendance(student, clf, reg, scaler)
    rows.append({
        "Student": name,
        "Current %": r["current_attendance"],
        "Predicted %": r["predicted_attendance"],
        "Risk": r["risk_level"],
        "Chance of falling short": f"{r['risk_probability_pct']}%",
        "Can still miss": r["can_skip"],
        "Must attend in a row": r["must_attend"],
    })
print(pd.DataFrame(rows).to_string(index=False))

## Notes & Next Steps

- **Using real data**: put your records in a CSV with the columns
  `Lectures_Delivered, Lectures_Attended, Medical_Leave, Duty_Leave,
  Future_Attendance` (the student's actual attendance % in the
  *next* period) and, optionally, `Sufficient_Attendance` (1/0). Then run
  `python attendance_predictor.py --data my_records.csv`. If
  `Sufficient_Attendance` is left out it is derived from
  `Future_Attendance`. Nothing else needs to change.
- **Required attendance**: `REQUIRED_ATTENDANCE` at the top of
  `attendance_predictor.py` defines what "sufficient" means -- it labels the
  synthetic data and is the boundary the classifier learns. If your
  institution's rule isn't 75%, change it and retrain.
- **Limitations**: four features and a linear/logistic model won't capture
  everything that affects a student's attendance (illness, family
  circumstances, timetable clashes), and the numbers above come from
  synthetic data, so they show the pipeline works -- not how accurate it
  will be on your students. Treat the output as an early-warning signal,
  not a verdict.
- **Extending it**: a natural next step beyond this baseline is a
  tree-based model (Random Forest / Gradient Boosting) to check whether
  non-linear feature interactions improve on it.